1. Price-based Features

In [ ]:
import pandas as pd

file_path = "C:/Users/vamsh/OneDrive/Desktop/Forex trading/Forex-Trading/config/forexrate/forex data/cleaned_eur_usd_10_years.csv"

data = pd.read_csv(file_path)

data.head(5)

In [2]:
#1.1 Price change (Daily Return)

data['Price Change'] = data['Close'].pct_change()  #percentage change

In [4]:
#1.2 Moving Averages - Capture trends

data['SMA_3'] = data['Close'].rolling(window=3).mean()
data['SMA_5'] = data['Close'].rolling(window=5).mean()
data['SMA_10'] = data['Close'].rolling(window=10).mean()

In [8]:
#1.3 Price Differences (Volatility Measure)

data['Price Difference'] = data['High'] - data['Low']

In [9]:
#1.4 - Percentage Price Change (Daily Returns)

data['Pct Price Change'] = (data['Close'].shift(-1) - data['Close']) / data['Close'] * 100

In [10]:
data.head(5)

,Date,Close,High,Low,Open,Price Change,SMA_3,SMA_5,SMA_10,Price Difference,Pct Price Change
0,2014-01-01,1.374495,1.377904,1.374400,1.374495,NaN,NaN,NaN,NaN,0.003504,0.158325
1,2014-01-02,1.376671,1.377467,1.363271,1.376595,0.001583,NaN,NaN,NaN,0.014197,-0.727065
2,2014-01-03,1.366662,1.367297,1.360170,1.366624,-0.007271,1.372609,NaN,NaN,0.007127,-0.516652
3,2014-01-06,1.359601,1.364610,1.357279,1.359582,-0.005167,1.367644,NaN,NaN,0.007331,0.264468
4,2014-01-07,1.363196,1.365799,1.359878,1.363066,0.002645,1.363153,1.368125,NaN,0.005921,-0.114382


2. Momentum Indicators

In [12]:
#2.1 - Relative Strength Index (RSI)
# RSI helps identify whether a market is overbought or oversold

def compute_rsi(data, timeperiod=14):
    delta = data['Close'].diff()  # Price change
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    
    avg_gain = gain.rolling(window=timeperiod).mean()
    avg_loss = loss.rolling(window=timeperiod).mean()
    
    rs = avg_gain / avg_loss
    rsi = 100 - (100 / (1 + rs))
    
    return rsi

data['RSI'] = compute_rsi(data, timeperiod=14)

In [13]:
"""
2.2 

Moving Average Convergence Divergence (MACD)
The MACD is calculated as the difference between a short-term (12-day) and 
a long-term (26-day) exponential moving average (EMA).
 The Signal Line is the 9-day EMA of the MACD.

"""

def compute_macd(data, fastperiod=12, slowperiod=26, signalperiod=9):
    fast_ema = data['Close'].ewm(span=fastperiod, adjust=False).mean()
    slow_ema = data['Close'].ewm(span=slowperiod, adjust=False).mean()

    macd = fast_ema - slow_ema
    
    macd_signal = macd.ewm(span=signalperiod, adjust=False).mean()
    
    return macd, macd_signal

data['MACD'], data['MACD Signal'] = compute_macd(data, fastperiod=12, slowperiod=26, signalperiod=9)

In [14]:
"""
2.3 
Stochastic Oscillator
The Stochastic Oscillator measures the current closing price relative to its range over a period of time 
(typically 14 days).

"""

def compute_stochastic_oscillator(data, fastk_period=14, slowk_period=3, slowd_period=3):
    low_min = data['Low'].rolling(window=fastk_period).min()
    high_max = data['High'].rolling(window=fastk_period).max()
    
    fast_k = 100 * (data['Close'] - low_min) / (high_max - low_min)
    
    # Slow %K (3-period smoothing of Fast %K)
    slow_k = fast_k.rolling(window=slowk_period).mean()
    
    # Slow %D (3-period smoothing of Slow %K)
    slow_d = slow_k.rolling(window=slowd_period).mean()
    
    return slow_k, slow_d

data['SlowK'], data['SlowD'] = compute_stochastic_oscillator(data, fastk_period=14, slowk_period=3, slowd_period=3)

In [17]:
data.head(10)

,Date,Close,High,Low,Open,Price Change,SMA_3,SMA_5,SMA_10,Price Difference,Pct Price Change,RSI,MACD,MACD Signal,SlowK,SlowD
0,2014-01-01,1.374495,1.377904,1.374400,1.374495,NaN,NaN,NaN,NaN,0.003504,0.158325,NaN,0.000000,0.000000,NaN,NaN
1,2014-01-02,1.376671,1.377467,1.363271,1.376595,0.001583,NaN,NaN,NaN,0.014197,-0.727065,NaN,0.000174,0.000035,NaN,NaN
2,2014-01-03,1.366662,1.367297,1.360170,1.366624,-0.007271,1.372609,NaN,NaN,0.007127,-0.516652,NaN,-0.000491,-0.000070,NaN,NaN
3,2014-01-06,1.359601,1.364610,1.357279,1.359582,-0.005167,1.367644,NaN,NaN,0.007331,0.264468,NaN,-0.001569,-0.000370,NaN,NaN
4,2014-01-07,1.363196,1.365799,1.359878,1.363066,0.002645,1.363153,1.368125,NaN,0.005921,-0.114382,NaN,-0.002109,-0.000718,NaN,NaN
5,2014-01-08,1.361637,1.363605,1.357202,1.361693,-0.001144,1.361478,1.365553,NaN,0.006403,-0.325733,NaN,-0.002633,-0.001101,NaN,NaN
6,2014-01-09,1.357202,1.363280,1.355489,1.357257,-0.003257,1.360679,1.361660,NaN,0.007791,0.258541,NaN,-0.003367,-0.001554,NaN,NaN
7,2014-01-10,1.360711,1.368701,1.357810,1.360711,0.002585,1.359850,1.360469,NaN,0.010890,0.475782,NaN,-0.003623,-0.001968,NaN,NaN
8,2014-01-13,1.367185,1.368476,1.363931,1.367222,0.004758,1.361699,1.361986,NaN,0.004545,-0.013672,NaN,-0.003267,-0.002228,NaN,NaN
9,2014-01-14,1.366998,1.369859,1.365029,1.367091,-0.000137,1.364965,1.362747,1.365436,0.004830,-0.010936,NaN,-0.002965,-0.002375,NaN,NaN


3. Volatility Indicators

In [18]:
"""
3.1 Bollinger Bands

Bollinger Bands measure volatility by calculating a range based on moving averages.

"""

def compute_bollinger_bands(data, timeperiod=20, nbdevup=2, nbdevdn=2):
    sma = data['Close'].rolling(window=timeperiod).mean()
    rolling_std = data['Close'].rolling(window=timeperiod).std()
    
    upper_band = sma + (nbdevup * rolling_std)
    lower_band = sma - (nbdevdn * rolling_std)
    
    return upper_band, sma, lower_band

data['BB_upper'], data['BB_middle'], data['BB_lower'] = compute_bollinger_bands(data, timeperiod=20, nbdevup=2, nbdevdn=2)

In [19]:
"""
3.2 Average True Range

ATR measures market volatility by calculating the true range, which is the maximum of:

The current high minus the current low
The absolute value of the current high minus the previous close
The absolute value of the current low minus the previous close
"""

def compute_atr(data, timeperiod=14):
    high_low = data['High'] - data['Low']
    high_close = (data['High'] - data['Close'].shift()).abs()
    low_close = (data['Low'] - data['Close'].shift()).abs()
    
    tr = high_low.combine(high_close, max).combine(low_close, max)
    
    atr = tr.rolling(window=timeperiod).mean()
    
    return atr

data['ATR'] = compute_atr(data, timeperiod=14)

In [20]:
data = data.dropna()

In [21]:
data.head(10)

,Date,Close,High,Low,Open,Price Change,SMA_3,SMA_5,SMA_10,Price Difference,Pct Price Change,RSI,MACD,MACD Signal,SlowK,SlowD,BB_upper,BB_middle,BB_lower,ATR
19,2014-01-28,1.367465,1.368940,1.363180,1.367596,-0.000971,1.368389,1.363061,1.361155,0.005761,-0.143377,55.441386,-0.002035,-0.003133,77.371776,59.081904,1.376500,1.363296,1.350091,0.008683
20,2014-01-29,1.365505,1.368599,1.360450,1.365430,-0.001434,1.367255,1.364979,1.361021,0.008149,0.025954,58.127448,-0.001699,-0.002846,72.345825,69.487427,1.375017,1.362846,1.350675,0.008708
21,2014-01-30,1.365859,1.366214,1.356061,1.365915,0.000260,1.366276,1.367306,1.361613,0.010153,-0.730821,55.371233,-0.001388,-0.002555,68.009538,72.575713,1.372725,1.362305,1.351886,0.008612
22,2014-01-31,1.355877,1.356239,1.348109,1.355859,-0.007308,1.362414,1.364700,1.361021,0.008131,-0.520638,39.006905,-0.001925,-0.002429,53.783703,64.713022,1.372352,1.361766,1.351181,0.009326
23,2014-02-03,1.348818,1.351960,1.347931,1.348818,-0.005206,1.356851,1.360705,1.360627,0.004029,0.273202,34.409145,-0.002886,-0.002520,33.410849,51.734697,1.373274,1.361227,1.349180,0.009548
24,2014-02-04,1.352503,1.354005,1.349564,1.352283,0.002732,1.352399,1.357712,1.360387,0.004440,-0.067586,38.400874,-0.003313,-0.002679,17.319391,34.837981,1.373307,1.360692,1.348077,0.009269
25,2014-02-05,1.351589,1.355300,1.350040,1.351662,-0.000676,1.350970,1.354929,1.359954,0.005260,0.142124,42.527668,-0.003683,-0.002880,11.888885,20.873042,1.373431,1.360190,1.346948,0.009070
26,2014-02-06,1.353510,1.362008,1.349109,1.353491,0.001421,1.352534,1.352459,1.359883,0.012899,0.421356,42.579522,-0.003777,-0.003059,18.006721,15.738332,1.373522,1.360005,1.346488,0.009534
27,2014-02-07,1.359213,1.364210,1.355579,1.358973,0.004214,1.354770,1.353126,1.358913,0.008631,0.200203,56.140936,-0.003353,-0.003118,26.756014,18.883873,1.373448,1.359930,1.346413,0.009522
28,2014-02-10,1.361934,1.365201,1.362027,1.362027,0.002002,1.358219,1.355750,1.358227,0.003174,0.196504,56.617187,-0.002765,-0.003047,40.245472,28.336069,1.372790,1.359668,1.346546,0.009625


In [23]:
"""
Target Variable

"""


# For Next Day Close
data.loc[:, 'Next Day Close'] = data['Close'].shift(-1)

# For Next Week Close
data.loc[:, 'Next Week Close'] = data['Close'].shift(-5)


In [24]:
data = data.dropna()  # Drop any remaining missing values
data.to_csv('C:/Users/vamsh/OneDrive/Desktop/Forex trading/Forex-Trading/config/forexrate/forex data/prepared_forex_data.csv', index=False)

In [25]:
data.head(10)

,Date,Close,High,Low,Open,Price Change,SMA_3,SMA_5,SMA_10,Price Difference,...,MACD,MACD Signal,SlowK,SlowD,BB_upper,BB_middle,BB_lower,ATR,Next Day Close,Next Week Close
19,2014-01-28,1.367465,1.368940,1.363180,1.367596,-0.000971,1.368389,1.363061,1.361155,0.005761,...,-0.002035,-0.003133,77.371776,59.081904,1.376500,1.363296,1.350091,0.008683,1.365505,1.352503
20,2014-01-29,1.365505,1.368599,1.360450,1.365430,-0.001434,1.367255,1.364979,1.361021,0.008149,...,-0.001699,-0.002846,72.345825,69.487427,1.375017,1.362846,1.350675,0.008708,1.365859,1.351589
21,2014-01-30,1.365859,1.366214,1.356061,1.365915,0.000260,1.366276,1.367306,1.361613,0.010153,...,-0.001388,-0.002555,68.009538,72.575713,1.372725,1.362305,1.351886,0.008612,1.355877,1.353510
22,2014-01-31,1.355877,1.356239,1.348109,1.355859,-0.007308,1.362414,1.364700,1.361021,0.008131,...,-0.001925,-0.002429,53.783703,64.713022,1.372352,1.361766,1.351181,0.009326,1.348818,1.359213
23,2014-02-03,1.348818,1.351960,1.347931,1.348818,-0.005206,1.356851,1.360705,1.360627,0.004029,...,-0.002886,-0.002520,33.410849,51.734697,1.373274,1.361227,1.349180,0.009548,1.352503,1.361934
24,2014-02-04,1.352503,1.354005,1.349564,1.352283,0.002732,1.352399,1.357712,1.360387,0.004440,...,-0.003313,-0.002679,17.319391,34.837981,1.373307,1.360692,1.348077,0.009269,1.351589,1.364610
25,2014-02-05,1.351589,1.355300,1.350040,1.351662,-0.000676,1.350970,1.354929,1.359954,0.005260,...,-0.003683,-0.002880,11.888885,20.873042,1.373431,1.360190,1.346948,0.009070,1.353510,1.363903
26,2014-02-06,1.353510,1.362008,1.349109,1.353491,0.001421,1.352534,1.352459,1.359883,0.012899,...,-0.003777,-0.003059,18.006721,15.738332,1.373522,1.360005,1.346488,0.009534,1.359213,1.359010
27,2014-02-07,1.359213,1.364210,1.355579,1.358973,0.004214,1.354770,1.353126,1.358913,0.008631,...,-0.003353,-0.003118,26.756014,18.883873,1.373448,1.359930,1.346413,0.009522,1.361934,1.367877
28,2014-02-10,1.361934,1.365201,1.362027,1.362027,0.002002,1.358219,1.355750,1.358227,0.003174,...,-0.002765,-0.003047,40.245472,28.336069,1.372790,1.359668,1.346546,0.009625,1.364610,1.369994
